# Results Calibration Lab

Purpose: diagnose where calibration/workload errors concentrate (matchup tiers, rest buckets, side pockets).

Use this after KPI WARNs to decide what to fix next.

Focused checks for concentrated miss pockets (matchup tier and long-rest workload).

In [1]:
from pathlib import Path
import sys
import polars as pl
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "production").exists() and (candidate / "src" / "Python").exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT / "src"))
from Python.exit_anomalies import apply_exit_anomaly_overrides

ODDS_DIR = ROOT / "artifacts" / "odds_log"
DECOMP_DAILY = ODDS_DIR / "k_error_decomposition_daily.parquet"
DECOMP_FULL = ODDS_DIR / "k_error_decomposition.parquet"
LEDGER_PATH = ROOT / "artifacts" / "odds_log" / "ledger.parquet"
EXIT_ANOMALY_PATH = ROOT / "production" / "ops" / "exit_anomaly_overrides.csv"

# Toggle active scope. False => all rows, True => anomaly-filtered core rows.
EXCLUDE_EXIT_ANOMALIES_FOR_PROCESS = False


def show_table(df: pl.DataFrame, max_rows: int = 30, height: int = 420):
    pdf = df.to_pandas().round(3)
    if len(pdf) <= max_rows:
        display(pdf)
        return
    table = pdf.to_html(index=False, na_rep="—")
    display(HTML(f"<div style='max-height:{height}px; overflow:auto; border:1px solid #4443; border-radius:6px'>{table}</div>"))


def _anomaly_ticket_ids() -> pl.DataFrame:
    if not LEDGER_PATH.exists():
        return pl.DataFrame({"ticket_id": []})
    ledger = pl.read_parquet(LEDGER_PATH)
    if ledger.is_empty():
        return pl.DataFrame({"ticket_id": []})
    tagged = apply_exit_anomaly_overrides(ledger, path=EXIT_ANOMALY_PATH)
    if "exit_anomaly_flag" not in tagged.columns:
        return pl.DataFrame({"ticket_id": []})
    return tagged.filter(pl.col("exit_anomaly_flag")).select("ticket_id").unique()


ANOMALY_TICKET_IDS = _anomaly_ticket_ids()
print("repo:", ROOT)
print(
    f"anomaly_ticket_ids={ANOMALY_TICKET_IDS.height} | "
    f"analysis_view={'core_ex_anomaly' if EXCLUDE_EXIT_ANOMALIES_FOR_PROCESS else 'all_rows'}"
)

repo: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props
anomaly_ticket_ids=6 | analysis_view=all_rows


In [2]:
if not DECOMP_DAILY.exists():
    print("Missing daily decomposition artifact.")
else:
    d = pl.read_parquet(DECOMP_DAILY)
    d = d.with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10).alias("game_date")) if "game_date" in d.columns else d
    scope_note = "daily artifact is pre-aggregated (all rows)"
    print(f"recent matchup-tier diagnostics | {scope_note}")
    tier_cols = [c for c in ["game_date", "matchup_tier", "n", "mae_err_k_rate", "bias_tbf"] if c in d.columns]
    if tier_cols:
        tier_view = d.select(tier_cols)
        if "game_date" in tier_view.columns:
            tier_view = tier_view.sort("game_date", descending=True)
        show_table(tier_view.head(30))

recent matchup-tier diagnostics | daily artifact is pre-aggregated (all rows)


,mae_err_k_rate
0,0.078
1,0.078
2,0.079
3,0.078
4,0.078
5,0.078
6,0.081
7,0.078
8,0.073
9,0.073


In [3]:
if not DECOMP_FULL.exists():
    print("Missing full decomposition artifact.")
else:
    k_all = pl.read_parquet(DECOMP_FULL)
    if ANOMALY_TICKET_IDS.height and "ticket_id" in k_all.columns:
        k_core = k_all.join(ANOMALY_TICKET_IDS, on="ticket_id", how="anti")
    else:
        k_core = k_all

    k = k_core if EXCLUDE_EXIT_ANOMALIES_FOR_PROCESS else k_all
    print(f"full decomposition rows | all={k_all.height} core={k_core.height} active={k.height}")

    rest_col = "days_rest" if "days_rest" in k.columns else None
    if rest_col is None:
        print("No days_rest column available in decomposition artifact.")
    else:
        rest = (
            k.with_columns(
                pl.when(pl.col(rest_col).cast(pl.Float64) >= 10)
                .then(pl.lit("long_rest_10_plus"))
                .otherwise(pl.lit("rest_under_10"))
                .alias("rest_bucket")
            )
            .group_by("rest_bucket")
            .agg(
                pl.len().alias("n"),
                pl.col("err_k_rate").abs().mean().alias("mae_err_k_rate") if "err_k_rate" in k.columns else pl.lit(None).alias("mae_err_k_rate"),
                pl.col("err_tbf").mean().alias("bias_tbf") if "err_tbf" in k.columns else pl.lit(None).alias("bias_tbf"),
            )
            .sort("rest_bucket")
        )
        show_table(rest)

    # Side-by-side scope check to quantify sensitivity.
    if "ticket_id" in k_all.columns:
        scope_comp = pl.concat(
            [
                k_all.with_columns(pl.lit("all_rows").alias("scope")),
                k_core.with_columns(pl.lit("core_ex_anomaly").alias("scope")),
            ],
            how="vertical_relaxed",
        )
        scope_summary = (
            scope_comp.group_by("scope")
            .agg(
                pl.len().alias("n"),
                pl.col("err_k_rate").abs().mean().alias("mae_err_k_rate") if "err_k_rate" in scope_comp.columns else pl.lit(None).alias("mae_err_k_rate"),
                pl.col("err_tbf").mean().alias("bias_tbf") if "err_tbf" in scope_comp.columns else pl.lit(None).alias("bias_tbf"),
            )
            .sort("scope")
        )
        print("scope comparison")
        show_table(scope_summary)

full decomposition rows | all=624 core=620 active=624


,rest_bucket,n,mae_err_k_rate,bias_tbf
0,long_rest_10_plus,51,0.093,-8.466
1,rest_under_10,573,0.076,0.279


scope comparison


,scope,n,mae_err_k_rate,bias_tbf
0,all_rows,624,0.078,-0.436
1,core_ex_anomaly,620,0.078,-0.515
